<a href="https://colab.research.google.com/github/chungweichang0918/baseball-tactics-agent/blob/main/notebooks/submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 多模態棒球戰術 Agent：看一段 MLB 精華，寫出有證據的戰術報告

**國立臺中科技大學 資工四乙　1111232077　張鈞崴**

> 執行方式：選單「執行階段 → 變更執行階段類型」選 **T4 GPU**，在左側鑰匙圖示新增密鑰 **`GEMINI_API_KEY`**，再按「執行階段 → 全部執行」。第三節會開出 Gradio 工作台，第四節以同一段示範影片自動跑完一次分析並顯示結果。

## 作業說明

### 1. 這個 Agent 要完成什麼任務

教練看比賽精華時想知道的不是「這球幾公里」，而是**下一個決定**：要不要盜壘、觸擊還是強攻？防守時要不要敬遠、怎麼配球？

這個 Agent 的輸入是：
- 一段 10–15 秒的投球或打擊影片
- 場上的局勢：局數、出局數、壘況、球數、比分、我方攻守
- 選用：從 Baseball Savant 下載的那場比賽逐球資料（Statcast CSV）

它的輸出是一份**每個數字都能追溯到證據**的戰術報告，共五節：
1. 結論
2. 影片觀察
3. RE24 數據
4. 推論
5. 信心與限制

### 2. 選擇的 Agent 類型：Plan-and-Execute ＋ Tool Use ＋ Reflection

| 模式 | 在這個 Agent 裡做什麼 | 為什麼這樣選 |
|---|---|---|
| **Plan-and-Execute** | Gemini 先依賽況與影片概況，產生一份結構化計畫（分析投手或打者、要回答的戰術問題、候選戰術、要呼叫的工具與順序），程式檢查通過後才執行 | 影片分析的工具有固定的前後關係（先骨架、再分段、再算指標），先排好整份計畫比 ReAct 一步一想更省呼叫，計畫本身也能被檢查 |
| **Tool Use** | 14 個工具：YOLO 骨架追蹤、OpenCV 鏡頭量測、動作分段、球軌跡、RE24 查表、Statcast 逐球數據、Gemini 看關鍵幀等 | 模型不自己算數字，時間、球速、得分期望值都交給確定性的程式 |
| **Reflection** | Critic 先用程式規則檢查（章節、證據引用、RE24 數字、戰術是否合法、mph／rpm 是否照抄 Statcast），再請 Gemini 檢查推論品質；不合格就退回 Synthesizer 修正，最多 2 次 | 規則保證數字不出錯，模型負責判斷推論有沒有誇大 |

### 3. 運作流程

```
影片＋賽況（＋Statcast CSV）
   │
   ├─ 偵察   probe_video 讀影片、scout_overview 讓 Gemini 判斷視角、statcast_pitch 讀入確認的那一球
   ├─ 規劃   Planner（Gemini）產生計畫 → 程式檢查工具、前置條件、戰術合法性 → 不合格重規劃
   ├─ 執行   Executor 依相依順序呼叫工具，每個結果存成一筆證據 E01、E02…；關鍵工具失敗時重新規劃一次
   ├─ 撰寫   Synthesizer（Gemini）依證據與關鍵幀寫報告，每項觀察都要引用證據編號
   └─ 審查   Critic：規則檢查 ＋ Gemini 檢查 → 不通過就附上原文片段退回修正
   │
戰術報告＋執行紀錄（計畫、證據、審查歷程）
```

### 4. 示範素材

- **影片**：2017 年世界大賽第 7 戰，9 局上，道奇隊 Alex Wood（左投）對太空人隊 Charlie Morton，15 秒，含兩球。[Google Drive](https://drive.google.com/file/d/1gsn8PCE5hf7v-rplgi-CdwL6lU3UnOLW/view?usp=sharing)
- **Statcast CSV**：同一場比賽（2017-11-01）的 294 筆逐球資料，從 Baseball Savant 下載。[Google Drive](https://drive.google.com/file/d/1vADaDWwt08kw0v0WBZOTGKC-t9zwqVki/view?usp=sharing)

MLB 轉播畫面有版權，這裡只作為課堂作業示範，不放進公開的程式碼庫。

## 一、環境準備

影像分析、RE24、Agent 的程式碼放在專案的 `bbagent` 套件裡，先從 GitHub 取得，再安裝固定版本的套件。

In [ ]:
REPO_URL = "https://github.com/chungweichang0918/baseball-tactics-agent.git"
PROJECT = "/content/baseball-tactics-agent"

import os
if not os.path.exists(PROJECT):
    !git clone -q {REPO_URL} {PROJECT}
%cd {PROJECT}

In [ ]:
%pip install -q -e . "gradio==6.31.0" "google-genai==2.33.0" "ultralytics==8.4.192" "lap==0.5.13"

安裝最後若出現 `google-colab … requires google-auth==2.49.0` 的紅字可以忽略，那是相容性提醒，不是安裝失敗。

`pip install -e` 要重新啟動 Python 才會生效，所以下一格直接把 `src` 加進匯入路徑。

In [ ]:
import sys
from pathlib import Path

SRC = Path(PROJECT) / "src"
os.chdir(PROJECT)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from bbagent.vision import device  # 必須在 torch 之前載入

from IPython.display import HTML, Image, Markdown, display
from bbagent.config import load_config

print("推論裝置：", device.resolve_device())
print("Gemini 型號：", load_config()["llm"]["fast_model"])

### 連接 Gemini

金鑰從 Colab 的 Secrets 讀取，不寫在程式裡。

In [ ]:
from getpass import getpass

api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY") or ""
except Exception:
    pass
if not api_key.strip():
    api_key = getpass("請輸入 Gemini API key（輸入不會顯示）：")
os.environ["GEMINI_API_KEY"] = api_key.strip()
del api_key
print("已設定金鑰" if os.environ["GEMINI_API_KEY"] else "未設定金鑰：Gemini 步驟會改用預設流程")

In [ ]:
!python scripts/download_models.py

from bbagent.llm.gemini_client import GeminiClient

client = GeminiClient()   # 429／5xx 自動重試；同樣的輸入重播快取，不重複消耗額度

## 二、示範素材與賽況

### 2-1. 下載影片與 Statcast CSV

In [ ]:
VIDEO_ID = "1gsn8PCE5hf7v-rplgi-CdwL6lU3UnOLW"
CSV_ID = "1vADaDWwt08kw0v0WBZOTGKC-t9zwqVki"
DATA = Path("/content/demo")
DATA.mkdir(exist_ok=True)
VIDEO = DATA / "ws2017_g7_wood_morton.mp4"
CSV = DATA / "savant_2017-11-01.csv"
for file_id, path in [(VIDEO_ID, VIDEO), (CSV_ID, CSV)]:
    if not path.exists():
        !gdown -q {file_id} -O {path}

from bbagent.vision.video import probe_video

meta = probe_video(VIDEO)
print(f"影片：{meta.width}×{meta.height}，{meta.fps:.2f} fps，{meta.duration_s:.1f} 秒，偵測到 {len(meta.scene_cuts)} 處切鏡")

### 2-2. 只分析投球的那一個鏡頭

轉播會切鏡。這段影片依序是：

| 鏡頭 | 時間 | 內容 |
|---|---|---|
| 01 | 0–3.5 秒 | 主審與打者特寫 |
| 02 | 3.5–8.1 秒 | **中外野視角，投出第一球** |
| 03 | 8.1–14.5 秒 | 打者特寫 |
| 04 | 14.5–15.3 秒 | 下一球的開頭 |

整段一起分析時，特寫畫面裡的人特別大，主體選擇會把**主審**誤認成投手（我實際測試時發生過）。所以只取鏡頭 02 分析，Gradio 工作台的「片段選取與鏡頭切換」也提供同樣的選擇。

In [ ]:
from bbagent.ui.studio import prepare_clip
from bbagent.ui.workspace import runs_root

edges = [0, *meta.scene_cuts, meta.frame_count]
shots = [(a / meta.fps, b / meta.fps) for a, b in zip(edges, edges[1:])]
for i, (a, b) in enumerate(shots, 1):
    print(f"鏡頭 {i:02d}：{a:.2f}–{b:.2f} 秒")
CLIP_START, CLIP_END = shots[1]
CLIP, OFFSET = prepare_clip(str(VIDEO), CLIP_START, CLIP_END, runs_root() / "clips")
print(f"分析片段：{CLIP_START:.2f}–{CLIP_END:.2f} 秒 → {CLIP.name}")

### 2-3. 從記分板讀出賽況

賽況以**我方觀點**輸入。這裡假設我方是防守的道奇隊，所以比分差是 1 − 5 = −4。記分板顯示：
- 9 局上、無人出局、壘上無人
- 球數 0-1
- 左投 Wood 對右打 Morton

In [ ]:
import cv2
from bbagent.ui.app import make_game
from bbagent.ui.diagram import situation_svg
from bbagent.ui.workspace import frame_at

display(Image(data=cv2.imencode(".jpg", cv2.cvtColor(frame_at(VIDEO, CLIP_START + 0.5), cv2.COLOR_RGB2BGR))[1].tobytes(), width=640))

# 局數, 上下半局, 出局, 壘上跑者, 壞球, 好球, 比分差, 我方攻守, 分析對象, 投球手, 打擊方向, 補充
GAME_VALUES = (9, "top", 0, [], 0, 1, -4, "defense", "pitcher", "L", "R",
               "2017 世界大賽第 7 戰：道奇 Alex Wood 投球，太空人 Charlie Morton 打擊")
game = make_game(*GAME_VALUES)
print(game.describe_zh())
display(HTML(situation_svg(game)))

### 2-4. 從 Statcast CSV 找出影片中的那一球

影片量不出轉速與擊球初速，球速也只能粗估，所以進階數據改由 Statcast 提供。

程式依賽況列出吻合度最高的候選球，再**由人對照影片確認**是哪一球，因為單靠一段剪輯無法保證自動找對。鏡頭 02 投出的是 0-1 時的伸卡球，主審判好球，對應第 70 打席第 2 球；下一球（指節曲球）三振打者。

In [ ]:
from bbagent.analysis.statcast import candidates, load_statcast, pitch_markdown

pitches = load_statcast(CSV)
print(f"CSV 共 {len(pitches)} 球；與賽況吻合度最高的候選：")
for p, diff in candidates(pitches, game, limit=5):
    print(f"  列 {p.row:>3}", "✓ 賽況相符" if not diff else "不同：" + "、".join(diff), "·", p.label())

pitch = next(p for p in pitches if p.get("at_bat_number") == "70" and p.get("pitch_number") == "2")
STATCAST = pitch.to_dict()
display(Markdown(pitch_markdown(STATCAST)))

## 三、Gradio 工作台

下一格會在這裡顯示工作台，同時產生一個臨時公開連結，只在這個 Colab 執行階段開著時有效。

**操作步驟**：
1. 「01 載入比賽精華」上傳影片，在「片段選取與鏡頭切換」選「鏡頭 02」，分析模式選「投球」。
2. 「02 比賽情境」填入賽況，並在「匯入 Statcast 逐球數據」上傳 CSV、選擇對應的那一球。
3. 按「開始分析精華」。

右側分頁會即時顯示：

| 分頁 | 內容 |
|---|---|
| 全場總覽 | 動作時間軸 |
| 投球分析／打擊分析 | 關鍵幀與動作曲線 |
| 逐球數據 | Statcast 數值與來源 |
| 守備模擬 | 手動情境推演，不是影片追蹤 |
| 分析報告 | 戰術報告與下載檔 |
| 執行紀錄 | Agent 的每一步：計畫、工具呼叫、審查 |

In [ ]:
from bbagent.ui.app import build_app, launch_options

if "demo" in globals():
    demo.close()
demo = build_app()
demo.queue(max_size=8, default_concurrency_limit=1)
demo.launch(share=True, inline=True, debug=False, show_error=False, **launch_options(demo))

## 四、Gradio 的對話結果

按下「開始分析精華」時，Gradio 呼叫的是 `analyze_workspace()`。這裡用同一個函式、同一段影片與賽況直接執行一次，讓結果留在筆記本裡。公開連結過期之後，這些結果仍然看得到。

### 4-1. 執行分析

In [ ]:
from bbagent.ui.workspace import analyze_workspace

last, final = None, None
for snap in analyze_workspace(str(VIDEO), CLIP_START, CLIP_END, "pitcher", -1, -1,
                              *GAME_VALUES, STATCAST, False, False):
    if snap[0] != last:
        print(snap[0])
        last = snap[0]
    final = snap
status, report, files, plan_table, chat = final[0], final[10], final[12], final[13], final[14]

### 4-2. 執行紀錄（Gradio「執行紀錄」分頁的對話內容）

Agent 每一步的思考、工具呼叫與結果、審查意見，依序列出。

In [ ]:
for m in chat:
    title = (m.get("metadata") or {}).get("title", "")
    display(Markdown(f"**{title}**\n\n{m['content']}" if title else m["content"]))

### 4-3. 計畫與關鍵幀

計畫表列出每個步驟用的工具、目的、依賴與執行狀態。關鍵幀上的綠框是分析對象，骨架左側畫藍色、右側畫橘色。

In [ ]:
display(Markdown(plan_table))
for path, caption in (final[6] or [])[:6]:
    print(caption)
    display(Image(filename=path, width=420))

### 4-4. 戰術報告（Gradio「分析報告」分頁）

報告的五節由 Synthesizer 撰寫並通過 Critic 審查。第 6 節的執行摘要由程式產生，最後附上這一球的 Statcast 數據表。

In [ ]:
display(Markdown(report))
print("下載檔：", *[Path(f).name for f in files], sep="\n- ")

## 五、Reflection：Critic 擋下錯誤數字

`demo_fault=True` 會在第一版報告刻意插入一個錯誤的 RE24 數字（ΔRE +0.999）。Critic 的規則檢查會發現它不是 RE24 工具算出的值，附上原文片段退回 Synthesizer 修正。

前一節已經呼叫過的 Gemini 輸入會重播快取，這一格只有修正與審查需要新的呼叫。

In [ ]:
from bbagent.agent.orchestrator import run_agent

for ev in run_agent(CLIP, game, client=client, demo_fault=True, statcast=STATCAST):
    if ev.kind == "critique" or (ev.kind == "thought" and ("注入" in ev.title or "退回" in ev.title)):
        print(ev.title)
        if ev.content:
            print("   " + ev.content.replace("\n", "\n   "))
    if ev.kind == "final":
        print(ev.title)

## 六、真實轉播上的準確度：揮棒偵測評估

為了知道打者分支在 MLB 轉播畫面上可不可靠，我用 [MLB-YouTube 資料集](https://github.com/piergiaj/mlb-youtube)（Piergiovanni & Ryoo, 2018）的人工標註做了評估。評估方法：
- 有揮棒與沒揮棒的片段各抽一半。
- 用「平衡準確率」＝（召回率＋特異度）÷ 2。
- **在第 5 戰校準門檻，在沒看過的第 7 戰驗證**，驗證資料不拿來調參數。

| 策略 | 做法 | 第 7 戰平衡準確率（71 段） |
|---|---|---|
| agent | 原本的主體選擇：分數最高的人 | 52.1% |
| broadcast | 排除畫面中最大的人（通常是投手） | 63.4% |
| timed | 用投手出手時間定出揮棒時窗，檢查雙手速度與移動距離 | **68.4%**（精確率 60.5%、召回率 74.2%） |

timed 的門檻是在第 5 戰校準出來的（76 段，平衡準確率 73.7%）：雙手速度 ≥ 1.6、雙手中點移動 ≥ 0.2。到了第 7 戰下降到 68.4%。

兩場比賽沒揮棒時的手速中位數不同（1.28 對 1.82），表示不同比賽、不同轉播的資料分布有差異，還需要第三場比賽才能確認門檻是否穩定。

## 七、心得

**1. 讓模型做判斷，讓程式做計算與檢查。**
一開始我以為 Agent 就是讓 Gemini 什麼都做。實際寫下來才發現，最容易出錯的正是數字：模型會把 ΔRE 四捨五入成別的值，也會把「得分期望值」和「得分期望值的變化」混在一起。所以 RE24 數字只能照抄工具結果，再由 Critic 的規則逐一比對。第五節的示範中，錯誤的 +0.999 一定會被擋下。後來接入 Statcast 時也用同一套做法：報告裡每個 mph、rpm 都要對得上那一球的數據。

**2. 真實資料會推翻在範例影片上的信心。**
做這份示範時，我第一次把整段 15 秒的轉播丟進去，關鍵幀上的綠框竟然框住了**主審**。原因是轉播在投球前後切到特寫，特寫裡的人比投手大很多，主體選擇的分數就被拉走了。只取中外野視角的那一個鏡頭後，才正確選到投手 Wood。這讓我理解為什麼工作台要提供「依切鏡選取」，也理解關鍵幀為什麼一定要讓人看過再相信。


揮棒偵測在自己拍的側拍影片上看起來沒問題。換成 MLB 轉播後，原本的方法平衡準確率只有 52%，幾乎是猜的。我把每一段誤判的原因列出來：
- 選錯人：選到投手而不是打者
- 手沒進畫面
- 速度門檻太嚴

針對這些原因改成「依投手出手時間找揮棒時窗」，並且嚴格分開校準用的比賽和驗證用的比賽，最後在沒看過的第 7 戰得到 68.4%。這個數字不漂亮，但它是誠實的；如果在同一場比賽上調參數，數字會好看很多，卻沒有意義。

**3. 知道影片量不到什麼，和量到什麼一樣重要。**
單一轉播鏡頭沒有相機校正：球速只能用「假設投手身高 1.8 公尺」換算來粗估，轉速和擊球初速則完全量不出來。與其讓模型「看起來」算出轉速，不如匯入 Statcast 逐球數據，並在每個數字旁標示來源：
- Statcast 追蹤
- 官方估計值
- 影片粗估

對應的那一球也由人確認，報告必須寫出「只有 1 球」的限制。

**4. 介面也是 Agent 的一部分。**
教練看不懂執行紀錄，Agent 做得再好也沒用。Gradio 的主題在淺色模式下會讓表格變白、左右兩欄各自捲動會互相干擾，這些小問題都會讓人看不下去。最後的工作台把計畫、證據、關鍵幀、報告與審查歷程都攤開，讓使用者能追查每個結論的依據。

**還可以改進的地方**：
- 用第三場比賽驗證揮棒門檻。
- 加入相機校正，讓球速估計更可靠。
- 讓 Statcast 自動依影片時間對齊候選球，減少人工確認。

## 參考資料

- Wang et al., 2023, [Plan-and-Solve Prompting](https://arxiv.org/abs/2305.04091)；Yao et al., 2023, [ReAct](https://arxiv.org/abs/2210.03629)
- Madaan et al., 2023, [Self-Refine](https://arxiv.org/abs/2303.17651)；Shinn et al., 2023, [Reflexion](https://arxiv.org/abs/2303.11366)
- Tom Tango, [RE24 得分期望值表](https://tangotiger.net/re24.html)（MLB 2010–2015）
- Baseball Savant, [Statcast Search](https://baseballsavant.mlb.com/statcast_search) 與 [CSV 欄位說明](https://baseballsavant.mlb.com/csv-docs)
- Piergiovanni & Ryoo, 2018, [MLB-YouTube 資料集](https://github.com/piergiaj/mlb-youtube)
- [Ultralytics YOLO](https://docs.ultralytics.com/)、[Google Gen AI SDK](https://googleapis.github.io/python-genai/)、[Gradio](https://www.gradio.app/)
- 專案原始碼：[github.com/chungweichang0918/baseball-tactics-agent](https://github.com/chungweichang0918/baseball-tactics-agent)

展示結束後，請執行下一格或中斷並刪除執行階段，關閉公開連結。

In [ ]:
demo.close()
print("工作台已關閉。")